# GraphHopper with a custom node kernel

Graph kernels in GraKeL are usually built for **one kind** of node
meta-information: the Weisfeiler-Lehman family compares *discrete labels*,
while GraphHopper or Propagation in their attribute mode compare *continuous
feature vectors*. Many real graphs carry **both**: a semantic label and a
continuous embedding per node. [Issue #118](
https://github.com/ysig/GraKeL/issues/118) asks exactly for that, with a node
kernel of the form

$$
k_{\text{node}}(u, v) = \frac{1}{2}\left(
k_{\text{label}}(\ell_u, \ell_v) + k_{\text{feat}}(f_u, f_v)\right),
\qquad
k_{\text{feat}}(f_u, f_v) = e^{-\lVert f_u - f_v \rVert}.
$$

`GraphHopper` accepts such a kernel through its
`node_kernel` parameter. GraphHopper decomposes into a **weighted sum of node
kernels**, where the weight of a pair of nodes counts how many times the two
nodes appear at the same hop of two equal-length shortest paths:

$$
k(G, G') = \sum_{v \in V}\, \sum_{v' \in V'} w(v, v')\, k_{\text{node}}(v, v')
$$

The built-in variants (`kernel_type` `'linear'`, `'gaussian'`, `'bridge'`)
choose a specific $k_{\text{node}}$ over numeric attribute matrices. With
`node_kernel`, **you** define $k_{\text{node}}$, and GraKeL hands you the
label values of the two nodes *exactly as they were stored on the graph* --
tuples mixing a discrete label with a feature vector included, no one-hot
concatenation needed.

In [1]:
import numpy as np

from grakel import GraphHopper
from grakel import GraphKernel

## The data: nodes with a label *and* a feature vector

We keep the graphs tiny and, importantly, give **all of them the same
structure** (a triangle). With identical shortest paths everywhere, the
weights $w(v, v')$ are the same for every pair of graphs, so the differences
between kernel values below come from the node kernel alone -- which makes
the example easy to read.

Each node carries a tuple `("label", features)`:

* graph **A**: the reference,
* graph **A'**: same labels, features perturbed by a little noise,
* graph **B**: same features, *different* labels,
* graph **C**: same labels, features scaled away.

In [2]:
def triangle(values):
    """A triangle graph whose nodes carry (label, feature) pairs."""
    adjacency = np.array([[0.0, 1.0, 1.0],
                          [1.0, 0.0, 1.0],
                          [1.0, 1.0, 0.0]])
    return (adjacency, {i: value for i, value in enumerate(values)})


rng = np.random.default_rng(42)

base = [("C", np.array([1.0, 0.0])),
        ("N", np.array([0.0, 1.0])),
        ("O", np.array([1.0, 1.0]))]

graphs = {
    "A":  triangle(base),
    "A'": triangle([(label, f + 0.01 * rng.standard_normal(2))
                    for label, f in base]),
    "B":  triangle([("X", f) for label, f in base]),
    "C":  triangle([(label, 3.0 * f) for label, f in base]),
}
print(graphs["A"][1])

{0: ('C', array([1., 0.])), 1: ('N', array([0., 1.])), 2: ('O', array([1., 1.]))}


## The node kernel

The callable receives two node label values and returns a number. Here it is
the kernel asked for in issue #118: half a delta kernel on the labels, half a
exponential similarity on the features. The built-in `'gaussian'`
kernel instead uses the squared Euclidean distance.

GraKeL makes you two promises about the values it passes in: they arrive as
stored (no numeric re-encoding), and each value you return is **validated**
-- it must be a real, finite number. It is your job to make the callable
*symmetric*; checking that explicitly, as below, is a good habit.

In [3]:
def node_kernel(u, v):
    """Average label equality and exponential feature similarity."""
    label_u, feat_u = u
    label_v, feat_v = v
    return 0.5 * (float(label_u == label_v)
                  + np.exp(-np.linalg.norm(feat_u - feat_v)))


# sanity checks of the callable itself, before handing it to GraKeL
values = [value for g in graphs.values() for value in g[1].values()]
for u in values:
    for v in values:
        k_uv = node_kernel(u, v)
        assert k_uv == node_kernel(v, u)          # symmetric
        assert np.isreal(k_uv) and np.isfinite(k_uv)  # real and finite
print("node_kernel is symmetric, real and finite")

node_kernel is symmetric, real and finite


## Computing the kernel matrix

`GraphHopper` is a scikit-learn transformer like every GraKeL kernel:
`fit_transform` computes the matrix on a training set, `transform` compares a
new set against it, and `normalize=True` divides each entry by the geometric
mean of the two graph self-similarities. Positive self-similarities then give
ones on the training matrix diagonal.

In [4]:
gh = GraphHopper(node_kernel=node_kernel, normalize=True)

names = list(graphs)
K = gh.fit_transform([graphs[name] for name in names])

header = "        " + "".join(f"{name:>8}" for name in names)
print(header)
for name, row in zip(names, K):
    print(f"{name:>8}" + "".join(f"{value:8.3f}" for value in row))

               A      A'       B       C
       A   1.000   0.994   0.470   0.523
      A'   0.994   1.000   0.466   0.522
       B   0.470   0.466   1.000   0.073
       C   0.523   0.522   0.073   1.000


Reading the matrix, row by row:

* **A vs A'** is close to 1: same labels, nearly same features.
* **A vs B** loses the entire label half of the node kernel (every label
  differs), and keeps only the feature half.
* **A vs C** has the same label contribution as A vs A before normalization,
  but the features are farther apart, so the feature contribution shrinks.

The diagonal is exactly 1 because of `normalize=True`.

## Train and test, and the `GraphKernel` wrapper

`fit` followed by `transform` compares new graphs against the fitted ones --
here the test graphs are members of the training families, so the columns
read like the rows above. The same kernel is also available through the
generic wrapper, as a parameter of the `graph_hopper` kernel dictionary.

In [5]:
train = [graphs["A"], graphs["A'"], graphs["B"]]
test = [graphs["A"], graphs["C"]]

gh = GraphHopper(node_kernel=node_kernel, normalize=True)
K_train = gh.fit_transform(train)
K_test = gh.transform(test)
print(K_test.round(3))

gk = GraphKernel(kernel={"name": "graph_hopper", "node_kernel": node_kernel},
                 normalize=True)
print("wrapper agrees with the direct class:",
      np.allclose(gk.fit_transform(train), K_train))

[[1.    0.994 0.47 ]
 [0.523 0.522 0.073]]
wrapper agrees with the direct class: True


## When a node kernel misbehaves

Because `fit` only parses the graphs, your callable runs for the first time
inside `fit_transform` or `transform`. Every returned value is checked
there, and a bad one is reported **with the node pair that produced it** --
you do not have to dig through shape mismatches deep inside the kernel
arithmetic.

A classic mistake is forgetting to reduce a feature comparison to a scalar
(the `np.exp` below is missing a norm):

In [6]:
def broken_kernel(u, v):
    """Forgets to reduce the feature comparison to a scalar."""
    label_u, feat_u = u
    label_v, feat_v = v
    return 0.5 * (float(label_u == label_v) + np.exp(-(feat_u - feat_v)))


try:
    GraphHopper(node_kernel=broken_kernel).fit_transform(train)
except TypeError as error:
    print(error)

node_kernel must return a real number for every pair of nodes, got array([1., 1.]) of type ndarray while comparing the node at position 0 (('C', array([1., 0.]))) with the node at position 0 (('C', array([1., 0.]))).


The same validation catches non-finite values (`NaN`, infinities) as
`ValueError`, and exceptions raised inside your callable are re-raised as
`TypeError` with the offending node pair, keeping your original traceback as
the cause.

## Remarks

* **Defaults are untouched.** Without `node_kernel`, `GraphHopper` applies
  its built-in node kernels through `kernel_type`. With `node_kernel`,
  leave `kernel_type` at its default (`'linear'`).
* **Any label values work.** Plain discrete labels with a delta kernel
  (`lambda u, v: float(u == v)`) are enough -- a mode the built-in
  attribute kernels never offered.
* **Cost.** The built-in variants evaluate all node pairs with one matrix
  product; a Python callable is invoked once per node pair per graph pair.
  For large graphs, keep the callable cheap.
* **Positive semi-definiteness** of the resulting matrix is your
  responsibility: it holds whenever the callable is itself a kernel
  (symmetric and positive semi-definite), as both halves of the issue #118
  kernel are.
* **Parallelism.** With `n_jobs` set, the callable may run from several
  threads; keep it stateless or thread-safe.

The full parameter documentation lives on the
[GraphHopper kernel page](https://ysig.github.io/GraKeL/kernels.html#graph-hopper-kernel)
of the GraKeL documentation.